# Large Language Models

# Example 1

In [1]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="tiiuae/falcon-rw-1b",
)

result = pipe(
    "Question: What is the capital of France?\nAnswer:",
    max_new_tokens=50
)

print(result[0]["generated_text"])

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the capital of France?
Answer: Paris.
Question: What is the capital of the United Kingdom?
Answer: London.
Question: What is the capital of Canada?
Answer: Ottawa.
Question: What is the capital of Australia?
Answer: Canberra.



# Example 2

In [2]:
!pip install -U transformers

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

input_text = "Write a short poem about Paris"

inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


i love the way i feel when i walk through the streets of paris .


In [12]:
input_text = """
You are a strict AI.
If the answer is unknown, reply ONLY with: I don't know.

Question: Who won yesterday IPL match?
Answer:
"""

inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=10)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

I don't know


# HuggingFace

In [13]:
!pip install huggingface_hub

In [19]:
from langchain_huggingface import HuggingFaceEndpoint

In [41]:
from langchain_huggingface import HuggingFaceEndpoint
import os

llm = HuggingFaceEndpoint(
    repo_id = 'google-bert/bert-base-uncased',
    task='text-generation',
    temperature = 1,
    max_new_tokens = 64, # Using max_new_tokens instead of max_length as per common practice for HuggingFaceEndpoint
    huggingfacehub_api_token = ""YOUR_HF_API_KEY""
)

In [42]:
from langchain_huggingface import HuggingFaceEndpoint
import os

# Redefine llm with a text generation model known to work well with 'text-generation' task
llm = HuggingFaceEndpoint(
    repo_id = 'distilgpt2', # Changed to a widely supported text generation model
    task='text-generation',
    temperature = 1,
    max_new_tokens = 64,
    huggingfacehub_api_token = ""YOUR_HF_API_KEY""
)

try:
    response = llm.invoke('I want a poem on paris')
    print(response)
except StopIteration:
    print("Error: Could not find an inference provider for the specified model and task. ")
    print("This often indicates an issue with the Hugging Face API token or temporary service unavailability.")
    print("Please ensure your 'huggingfacehub_api_token' is valid and has access to the Hugging Face Inference API.")

Error: Could not find an inference provider for the specified model and task. 
This often indicates an issue with the Hugging Face API token or temporary service unavailability.
Please ensure your 'huggingfacehub_api_token' is valid and has access to the Hugging Face Inference API.


In [43]:
# Load model directly

# !pip install transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# Prompt Templates

In [44]:
x = 11

print(f'the vaule x is : {x}')

the vaule x is : 11


In [50]:
!pip install langchain
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["cuisine"],
    template="I want to open a restaurant for {cuisine} food. Suggest names."
)

prompt = prompt_template.format(cuisine="Indian")

print(prompt)

I want to open a restaurant for Indian food. Suggest names.


In [51]:
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=50)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

asian fusion


# Chains

Combine LLMs and Prompts in a multi step workflow



 Chain no - 1, which is good in solving math problems
 Chain no - 2, which is good at translation



 Take math question ----> Chain1(GPT) ------>Solution in English ------> Chain 2(Gemini) ---> Translatet it to any language


In [66]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["product"],
    template="""
You are a professional business branding expert.

Suggest 3 company names for a {product} business.

STRICT RULES:
- Names must be safe and appropriate
- No harmful, drug-related, or offensive words
- Must sound like real company/brand names
- All names must be different
- Output only names separated by commas

Answer:
"""
)

In [67]:
def run_chain(input_value):
    prompt = prompt_template.format(product=input_value)

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=40,
    top_p=0.9
)

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

In [68]:
response = run_chain("Milk")
print(response)

Milk, Milkshake, Milkshake


# Combine chains and create a sequence using SimpleSequentialChain

In [70]:
from langchain_core.prompts import PromptTemplate

prompt1 = PromptTemplate(
    input_variables=["cuisine"],
    template="Suggest a restaurant name for {cuisine} food"
)

In [71]:
prompt2 = PromptTemplate(
    input_variables=["restaurant_name"],
    template="Suggest menu items for {restaurant_name}"
)

In [78]:
def chain1(cuisine):
    prompt = prompt1.format(cuisine=cuisine)

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=30)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def chain2(restaurant_name):
    prompt = prompt2.format(restaurant_name=restaurant_name)

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [79]:
def sequential_chain(cuisine):
    name = chain1(cuisine)
    menu = chain2(name)

    return name, menu

In [80]:
name, menu = sequential_chain("Indian")

print("Restaurant Name:", name)
print("Menu Items:", menu)

Restaurant Name: saigon
Menu Items: fried rice, fried chicken, fried rice, fried chicken, fried rice, fried chicken, fried rice, fried chicken, fried chicken, fried chicken, fried chicken, fried chicken, fried


cuisine name (user input) ----> Prompt (using prompt template name) ---> LLM (for getting suggestion on restaurant names)  


resturant_name (given by LLM) ----> Prompt 2 (items) ---> LLM (for getting menu items for the restaurant)

# AI AGENTS Mini Project

In [81]:
!pip install google-search-results

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=1fa41d25221cfef9eb92af28f5639079a9fd04ce6b5a73ef7fd901df2f890c37
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


# SerpAPI - it is a realtime API to access GOOGLE search results

In [83]:
os.environ['SERPAPI_API_KEY'] = 'YOUR_SERPAPI_API_KEY'

In [86]:
!pip install langchain langchain-community langchain-core

In [91]:
!pip install google-search-results

import os
from serpapi import GoogleSearch
from transformers import pipeline

# 🔑 API KEY
os.environ["SERPAPI_API_KEY"] = "YOUR_SERPAPI_API_KEY"

# 🔹 Local model
pipe = pipeline("text-generation", model="gpt2")

# 🔍 SerpAPI search function
def search_tool(query):
    params = {
        "engine": "google",
        "q": query,
        "api_key": os.environ["SERPAPI_API_KEY"]
    }

    search = GoogleSearch(params)
    results = search.get_dict()

    if "organic_results" in results:
        return results["organic_results"][0]["snippet"]
    else:
        return "No results found"


# 🤖 FINAL AGENT (WORKING)
def agent(query):
    if "ipl" in query.lower() or "who won" in query.lower():
        return search_tool(query)
    else:
        result = pipe(query, max_new_tokens=50)
        return result[0]["generated_text"]


# 🚀 TEST
print(agent("Who won yesterday IPL match?"))
print(agent("Write a poem about Paris"))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Match 24 · APR, THU 16 , 7:30 pm IST. Punjab Kings Won by 7 Wickets · 195/6. (20.0 OV) ; Match 23 · APR, WED 15 , 7:30 pm IST. Royal Challengers Bengaluru Won by 5 ...
Write a poem about Paris.

The poem that you'd like to write about Paris.

The poem that you'd like to write about Paris.

The poem that you'd like to write about Paris.

The poem that you'd like to write


In [92]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=a7a63eb9628d0022b229e9bdc8c01d2b9971a98d9be246bc6ab87d29528c0415
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [94]:
import wikipedia

# 🔢 Add tool
def add(input):
    numbers = [int(s) for s in input.split() if s.isdigit()]
    return sum(numbers)

# 📚 Wikipedia tool
def wiki_tool(query):
    try:
        return wikipedia.summary(query, sentences=2)
    except:
        return "No information found"

In [95]:
def agent(query):

    # 🔢 Use add tool
    if "add" in query.lower():
        return add(query)

    # 📚 Use wikipedia
    elif "who" in query.lower() or "what" in query.lower():
        return wiki_tool(query)

    # 🤖 Use LLM
    else:
        result = pipe(
            query,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.9,
            repetition_penalty=1.5
        )
        return result[0]["generated_text"]

In [96]:
print(agent("add 10 and 100"))
print(agent("Who is Virat Kohli"))
print(agent("Write a poem about Paris"))

110


Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Virat Kohli (born 5 November 1988) is an Indian international cricketer and the former all-format captain of the Indian national cricket team. He is a right-handed batter and occasional right-arm medium pace bowler.
Write a poem about Paris
In the last two years, I've come to realize that my whole world is becoming very busy. Sometimes it just feels like reading an old favorite: "Paris's New 'One-Shoot City' - A Memoir." It gets us


### 📌 Summary

In this notebook, we explored the fundamentals of building AI applications using a local LLM. We started by understanding how to use a language model (GPT2/FLAN-T5) for basic text generation tasks like answering questions and generating poems.

Next, we learned about **Prompt Templates** to create dynamic and reusable prompts. We then implemented **Chains** and **Sequential Chains** to connect multiple steps and build structured workflows.

We moved on to **Agents**, where we combined the LLM with tools to handle different types of queries. Since modern LangChain agent APIs were not compatible with our setup, we implemented a **custom agent**.

The agent integrates:

* A **calculation tool** for arithmetic operations
* A **Wikipedia tool** for factual information
* A **local LLM** for general responses

Finally, we improved output quality by handling issues like hallucination, repetition, and formatting using prompt engineering and generation parameters.

### ✅ Conclusion

This notebook demonstrates how to build a simple AI agent system by combining LLMs with tools, providing a strong foundation for real-world AI applications without using paid APIs.
